In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix, hstack, identity, eye
from lightfm import LightFM
from lightfm.evaluation import precision_at_k as lfm_precision_at_k
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
np.random.seed(42)

/opt/homebrew/Caskroom/miniconda/base/envs/data_analysis/lib/python3.11/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [2]:
df = pd.read_csv('../../data/children_products/children_product_cleaned_full_year.csv', low_memory=False)
print(f"Размер исходного датасета: {df.shape}")

Размер исходного датасета: (4050261, 17)


In [3]:
df_filtered = df[(df['Статус'] == 'Доставлен') & (df['Отменено'] == 'Нет')].copy()
df_filtered = df_filtered.dropna(subset=['Телефон_new', 'ID_SKU', 'Дата'])
df_filtered['Дата'] = pd.to_datetime(df_filtered['Дата'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['Дата'])

# Конвертация числовых колонок
for col in ['Цена', 'Маржа', 'СуммаУслуг']:
    df_filtered[col] = df_filtered[col].astype(str).str.replace(' ', '').str.replace(',', '.')
    df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce')

print(f"После фильтрации (доставленные): {df_filtered.shape}")
print(f"Диапазон дат: {df_filtered['Дата'].min()} — {df_filtered['Дата'].max()}")

После фильтрации (доставленные): (2053127, 17)
Диапазон дат: 2017-01-01 17:38:00 — 2017-12-01 19:48:00


In [4]:
MIN_INTERACTIONS = 3

for _ in range(5):
    user_counts = df_filtered.groupby('Телефон_new').size()
    item_counts = df_filtered.groupby('ID_SKU').size()
    active_users = user_counts[user_counts >= MIN_INTERACTIONS].index
    active_items = item_counts[item_counts >= MIN_INTERACTIONS].index
    before = len(df_filtered)
    df_filtered = df_filtered[
        df_filtered['Телефон_new'].isin(active_users) &
        df_filtered['ID_SKU'].isin(active_items)
    ]
    if len(df_filtered) == before:
        break

n_users = df_filtered['Телефон_new'].nunique()
n_items = df_filtered['ID_SKU'].nunique()
print(f"После фильтрации (≥{MIN_INTERACTIONS} покупок):")
print(f"  Пользователей: {n_users:,}, Товаров: {n_items:,}")
print(f"  Взаимодействий: {len(df_filtered):,}")

После фильтрации (≥3 покупок):
  Пользователей: 129,421, Товаров: 71,502
  Взаимодействий: 1,694,548


In [5]:
interactions = df_filtered.groupby(['Телефон_new', 'ID_SKU']).size().reset_index(name='count')

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
interactions['user_id'] = user_encoder.fit_transform(interactions['Телефон_new'])
interactions['item_id'] = item_encoder.fit_transform(interactions['ID_SKU'])

interactions_with_date = df_filtered.merge(
    interactions[['Телефон_new', 'ID_SKU', 'user_id', 'item_id', 'count']],
    on=['Телефон_new', 'ID_SKU'], how='inner'
).sort_values('Дата')

split_date = interactions_with_date['Дата'].quantile(0.8)
print(f"Дата разделения: {split_date}")

train_data = interactions_with_date[interactions_with_date['Дата'] < split_date]
test_data = interactions_with_date[interactions_with_date['Дата'] >= split_date]

train_interactions = train_data.groupby(['user_id', 'item_id'])['count'].sum().reset_index()
test_interactions = test_data.groupby(['user_id', 'item_id'])['count'].sum().reset_index()

train_users = set(train_interactions['user_id'].unique())
test_users = set(test_interactions['user_id'].unique())
print(f"Train: {len(train_interactions):,} пар, Test: {len(test_interactions):,} пар")
print(f"Cold start users: {len(test_users - train_users):,}")

Дата разделения: 2017-09-06 11:48:00
Train: 1,207,042 пар, Test: 320,341 пар
Cold start users: 16,605


In [6]:
n_users_total = len(user_encoder.classes_)
n_items_total = len(item_encoder.classes_)

def build_sparse(df, n_users, n_items):
    return csr_matrix(
        (np.ones(len(df)),
         (df['user_id'].values, df['item_id'].values)),
        shape=(n_users, n_items)
    )

train_matrix = build_sparse(train_interactions, n_users_total, n_items_total)
test_matrix = build_sparse(test_interactions, n_users_total, n_items_total)

print(f"Train matrix: {train_matrix.shape}, ненулевых: {train_matrix.nnz:,}")
print(f"Test matrix:  {test_matrix.shape}, ненулевых: {test_matrix.nnz:,}")

Train matrix: (129421, 71502), ненулевых: 1,207,042
Test matrix:  (129421, 71502), ненулевых: 320,341


In [7]:
# Строим фичи только по train-данным, чтобы избежать утечки
user_phone_to_id = dict(zip(
    interactions['Телефон_new'], interactions['user_id']
))

# Агрегация по пользователям из train
user_agg = train_data.groupby('Телефон_new').agg(
    geo_mode=('Гео', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Регионы'),
    delivery_mode=('МетодДоставки', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown'),
    avg_price=('Цена', 'mean'),
    n_orders=('НомерЗаказаНаСайте', 'nunique'),
    n_unique_categories=('Группа2', 'nunique'),
    n_unique_items=('ID_SKU', 'nunique')
).reset_index()

user_agg['user_id'] = user_agg['Телефон_new'].map(user_phone_to_id)
user_agg = user_agg.dropna(subset=['user_id'])
user_agg['user_id'] = user_agg['user_id'].astype(int)

print(f"Пользователей с фичами: {len(user_agg):,}")
user_agg.head()

Пользователей с фичами: 112,816


,Телефон_new,geo_mode,delivery_mode,avg_price,n_orders,n_unique_categories,n_unique_items,user_id
0,32555749-545749525150 .,Москва,Магазины,207.800000,1,1,5,0
1,55515749-50495648505172,Регионы,DPD,1320.333333,1,1,15,2
2,55525753-49504848554873,Москва,Курьерская,81.000000,1,1,2,3
3,55525753-50505757515573,Москва,Курьерская,184.750000,1,2,4,4
4,55525753-51545355524977,Москва,Курьерская,525.666667,1,3,3,5


In [8]:
# Бинаризация среднего чека и кол-ва заказов по квантилям
user_agg['price_bin'] = pd.qcut(user_agg['avg_price'], q=3, labels=['price_low', 'price_mid', 'price_high'])
user_agg['orders_bin'] = pd.qcut(user_agg['n_orders'].rank(method='first'), q=3, labels=['orders_low', 'orders_mid', 'orders_high'])

# Группировка метода доставки (аналогично research_data)
def group_delivery(method):
    method = str(method)
    if 'Курьерская' in method:
        return 'delivery_courier'
    elif 'Магазины' in method or 'Самовывоз' in method:
        return 'delivery_store'
    else:
        return 'delivery_pickup'

user_agg['delivery_group'] = user_agg['delivery_mode'].apply(group_delivery)

# Гео-префикс
user_agg['geo_feature'] = 'geo_' + user_agg['geo_mode'].astype(str)

print("Распределение Гео:")
print(user_agg['geo_feature'].value_counts())
print("\nРаспределение Доставки:")
print(user_agg['delivery_group'].value_counts())
print("\nРаспределение ценового сегмента:")
print(user_agg['price_bin'].value_counts())

Распределение Гео:
geo_feature
geo_Регионы    49995
geo_Москва     49727
geo_МО         13094
Name: count, dtype: int64

Распределение Доставки:
delivery_group
delivery_store      79297
delivery_courier    28078
delivery_pickup      5441
Name: count, dtype: int64

Распределение ценового сегмента:
price_bin
price_low     37617
price_high    37606
price_mid     37593
Name: count, dtype: int64


In [9]:
# Собираем список фичей для каждого пользователя
user_feature_list = []

for _, row in user_agg.iterrows():
    features = [
        row['geo_feature'],
        row['delivery_group'],
        str(row['price_bin']),
        str(row['orders_bin'])
    ]
    user_feature_list.append((int(row['user_id']), features))

# Все уникальные фичи
all_user_features = sorted(set(
    f for _, feats in user_feature_list for f in feats
))
print(f"Уникальных user-фичей: {len(all_user_features)}")
print(all_user_features)

Уникальных user-фичей: 12
['delivery_courier', 'delivery_pickup', 'delivery_store', 'geo_МО', 'geo_Москва', 'geo_Регионы', 'orders_high', 'orders_low', 'orders_mid', 'price_high', 'price_low', 'price_mid']


In [10]:
item_sku_to_id = dict(zip(
    interactions['ID_SKU'], interactions['item_id']
))

# Агрегация по товарам из train
item_agg = train_data.groupby('ID_SKU').agg(
    group2_mode=('Группа2', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown'),
    group3_mode=('Группа3', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown'),
    type_mode=('Тип', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown'),
    avg_price=('Цена', 'mean')
).reset_index()

item_agg['item_id'] = item_agg['ID_SKU'].map(item_sku_to_id)
item_agg = item_agg.dropna(subset=['item_id'])
item_agg['item_id'] = item_agg['item_id'].astype(int)

# Ценовой сегмент товара
item_agg['item_price_bin'] = pd.qcut(
    item_agg['avg_price'], q=4, labels=['item_cheap', 'item_budget', 'item_mid', 'item_premium']
)

# Добавляем префиксы для уникальности
item_agg['group2_feat'] = 'g2_' + item_agg['group2_mode'].astype(str)
item_agg['group3_feat'] = 'g3_' + item_agg['group3_mode'].astype(str)
item_agg['type_feat'] = 'type_' + item_agg['type_mode'].astype(str)

print(f"Товаров с фичами: {len(item_agg):,}")
print(f"\nУникальных Группа2: {item_agg['group2_feat'].nunique()}")
print(f"Уникальных Группа3: {item_agg['group3_feat'].nunique()}")
print(f"Уникальных Тип: {item_agg['type_feat'].nunique()}")

Товаров с фичами: 68,523

Уникальных Группа2: 13
Уникальных Группа3: 92
Уникальных Тип: 5


In [11]:
# Собираем фичи для каждого товара
item_feature_list = []

for _, row in item_agg.iterrows():
    features = [
        row['group2_feat'],
        row['group3_feat'],
        row['type_feat'],
        str(row['item_price_bin'])
    ]
    item_feature_list.append((int(row['item_id']), features))

all_item_features = sorted(set(
    f for _, feats in item_feature_list for f in feats
))
print(f"Уникальных item-фичей: {len(all_item_features)}")
print(f"Примеры: {all_item_features[:10]}")

Уникальных item-фичей: 114
Примеры: ['g2_ДЕТСКОЕ ПИТАНИЕ', 'g2_ЖЕНСКИЕ ШТУЧКИ', 'g2_ИГРУШКИ', 'g2_КАНЦТОВАРЫ, КНИГИ, ДИСКИ', 'g2_КОСМЕТИКА/ГИГИЕНА', 'g2_КРУПНОГАБАРИТНЫЙ ТОВАР', 'g2_ОБУВЬ', 'g2_ПОДГУЗНИКИ', 'g2_СОПУТСТВУЮЩИЕ ТОВАРЫ', 'g2_ТЕКСТИЛЬ, ТРИКОТАЖ']


In [12]:
def build_feature_matrix(feature_list, all_features, n_entities):
    """
    Строит sparse feature-матрицу: identity (n_entities x n_entities) + side features.
    
    feature_list: [(entity_id, [feat1, feat2, ...]), ...]
    all_features: список всех уникальных фичей
    n_entities: кол-во сущностей (users или items)
    
    Возвращает: csr_matrix (n_entities, n_entities + len(all_features))
    """
    feat_to_idx = {f: i for i, f in enumerate(all_features)}
    n_feats = len(all_features)
    
    # Side features matrix (n_entities x n_features)
    rows, cols, data = [], [], []
    for entity_id, feats in feature_list:
        for f in feats:
            if f in feat_to_idx:
                rows.append(entity_id)
                cols.append(feat_to_idx[f])
                data.append(1.0)
    
    side_matrix = csr_matrix(
        (data, (rows, cols)),
        shape=(n_entities, n_feats)
    )
    
    # Identity + side features
    identity_matrix = eye(n_entities, format='csr')
    feature_matrix = hstack([identity_matrix, side_matrix], format='csr')
    
    return feature_matrix


user_features_matrix = build_feature_matrix(user_feature_list, all_user_features, n_users_total)
item_features_matrix = build_feature_matrix(item_feature_list, all_item_features, n_items_total)

print(f"User features matrix: {user_features_matrix.shape}")
print(f"  (n_users={n_users_total} + n_user_features={len(all_user_features)})")
print(f"Item features matrix: {item_features_matrix.shape}")
print(f"  (n_items={n_items_total} + n_item_features={len(all_item_features)})")

User features matrix: (129421, 129433)
  (n_users=129421 + n_user_features=12)
Item features matrix: (71502, 71616)
  (n_items=71502 + n_item_features=114)


## Обучение LightFM с фичами (WARP loss)

In [13]:
model = LightFM(
    no_components=64,
    loss='warp',
    learning_rate=0.05,
    item_alpha=1e-6,
    user_alpha=1e-6,
    random_state=42
)

NUM_EPOCHS = 20
NUM_THREADS = 1  # LightFM без OpenMP — 1 поток

train_precisions = []

for epoch in range(NUM_EPOCHS):
    model.fit_partial(
        train_matrix,
        user_features=user_features_matrix,
        item_features=item_features_matrix,
        epochs=1,
        num_threads=NUM_THREADS
    )
    p = lfm_precision_at_k(
        model, train_matrix,
        user_features=user_features_matrix,
        item_features=item_features_matrix,
        k=10, num_threads=NUM_THREADS
    ).mean()
    train_precisions.append(p)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}: train Precision@10 = {p:.4f}")

KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(train_precisions) + 1), train_precisions, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Precision@10 (train)')
plt.title('LightFM WARP + Features — кривая обучения')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def get_recommendations_lfm(model, user_id, train_matrix, n_items,
                            user_features=None, item_features=None, k=20):
    """Top-k рекомендаций для пользователя, исключая уже купленные."""
    scores = model.predict(
        user_id, np.arange(n_items),
        user_features=user_features,
        item_features=item_features
    )
    bought = set(train_matrix[user_id].indices)
    candidate_items = [i for i in np.argsort(-scores) if i not in bought]
    return candidate_items[:k]


def precision_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(rec_k) if rec_k else 0.0

def recall_at_k(recommended, relevant, k):
    rel = set(relevant)
    return len(set(recommended[:k]) & rel) / len(rel) if rel else 0.0

def map_at_k(recommended, relevant, k):
    rel = set(relevant)
    if not rel:
        return 0.0
    score, hits = 0.0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in rel:
            hits += 1.0
            score += hits / (i + 1.0)
    return score / min(len(rel), k)

def ndcg_at_k(recommended, relevant, k):
    rel = set(relevant)
    if not rel:
        return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in rel)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(rel), k)))
    return dcg / idcg if idcg > 0 else 0.0

In [ ]:
def evaluate_model(model, train_matrix, test_interactions, n_items,
                   user_features=None, item_features=None, k_values=[5, 10, 20]):
    test_user_items = test_interactions.groupby('user_id')['item_id'].apply(list).to_dict()
    train_users = set(np.where(train_matrix.getnnz(axis=1) > 0)[0])
    eval_users = [u for u in test_user_items if u in train_users]

    print(f"Оценка на {len(eval_users):,} пользователях...")

    metrics = {k: {'precision': [], 'recall': [], 'map': [], 'ndcg': []} for k in k_values}

    for user_id in eval_users:
        rec_items = get_recommendations_lfm(
            model, user_id, train_matrix, n_items,
            user_features=user_features,
            item_features=item_features,
            k=max(k_values)
        )
        relevant = test_user_items[user_id]
        for k in k_values:
            metrics[k]['precision'].append(precision_at_k(rec_items, relevant, k))
            metrics[k]['recall'].append(recall_at_k(rec_items, relevant, k))
            metrics[k]['map'].append(map_at_k(rec_items, relevant, k))
            metrics[k]['ndcg'].append(ndcg_at_k(rec_items, relevant, k))

    return {
        k: {m: float(np.mean(v)) for m, v in mv.items()}
        for k, mv in metrics.items()
    }

In [ ]:
results = evaluate_model(
    model, train_matrix, test_interactions, n_items_total,
    user_features=user_features_matrix,
    item_features=item_features_matrix
)

In [ ]:
print("=== LightFM WARP + Features Metrics (Full Year) ===")
for k in [5, 10, 20]:
    r = results[k]
    print(f"\nK={k}:")
    print(f"  Precision@{k}: {r['precision']:.4f}")
    print(f"  Recall@{k}:    {r['recall']:.4f}")
    print(f"  MAP@{k}:       {r['map']:.4f}")
    print(f"  NDCG@{k}:      {r['ndcg']:.4f}")

In [ ]:
metrics_df = pd.DataFrame(results).T
metrics_df.index.name = 'K'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(metrics_df.index, metrics_df['precision'], marker='o', linewidth=2)
axes[0, 0].set_xlabel('K'); axes[0, 0].set_ylabel('Precision@K')
axes[0, 0].set_title('Precision@K'); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(metrics_df.index, metrics_df['recall'], marker='o', linewidth=2, color='orange')
axes[0, 1].set_xlabel('K'); axes[0, 1].set_ylabel('Recall@K')
axes[0, 1].set_title('Recall@K'); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(metrics_df.index, metrics_df['map'], marker='o', linewidth=2, color='green')
axes[1, 0].set_xlabel('K'); axes[1, 0].set_ylabel('MAP@K')
axes[1, 0].set_title('MAP@K'); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(metrics_df.index, metrics_df['ndcg'], marker='o', linewidth=2, color='red')
axes[1, 1].set_xlabel('K'); axes[1, 1].set_ylabel('NDCG@K')
axes[1, 1].set_title('NDCG@K'); axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('LightFM WARP + Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Веса user-фичей (после identity-части)
user_embeddings = model.user_embeddings
user_feat_embeddings = user_embeddings[n_users_total:]  # skip identity part

user_feat_norms = np.linalg.norm(user_feat_embeddings, axis=1)
user_feat_importance = pd.DataFrame({
    'feature': all_user_features,
    'embedding_norm': user_feat_norms
}).sort_values('embedding_norm', ascending=False)

print("User feature importance (по норме эмбеддинга):")
print(user_feat_importance.to_string(index=False))

In [ ]:
# Веса item-фичей
item_embeddings = model.item_embeddings
item_feat_embeddings = item_embeddings[n_items_total:]  # skip identity part

item_feat_norms = np.linalg.norm(item_feat_embeddings, axis=1)
item_feat_importance = pd.DataFrame({
    'feature': all_item_features,
    'embedding_norm': item_feat_norms
}).sort_values('embedding_norm', ascending=False)

print(f"Item feature importance (top-20 по норме эмбеддинга):")
print(item_feat_importance.head(20).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# User features
ax = axes[0]
ufi = user_feat_importance
ax.barh(ufi['feature'], ufi['embedding_norm'])
ax.set_xlabel('Embedding Norm')
ax.set_title('User Feature Importance')
ax.invert_yaxis()

# Item features (top-20)
ax = axes[1]
ifi = item_feat_importance.head(20)
ax.barh(ifi['feature'], ifi['embedding_norm'])
ax.set_xlabel('Embedding Norm')
ax.set_title('Item Feature Importance (top-20)')
ax.invert_yaxis()

plt.tight_layout()
plt.show()